# Week 6 — Product Valuation & Deal Checker

A **product valuation** tool that estimates typical selling price from a description and labels it as **Budget**, **Mid-range**, or **Premium**. 

- **Single-market valuation** in **USD** (or your catalog currency)
- **Price band** classification (low / mid / high) from data-driven quartiles
- **Deal check**: quick verdict (e.g. "Likely fair", "Check similar listings")
- **Batch evaluation** with MAPE and band accuracy (not just MAE/R²)

Use case: sellers, buyers, or inventory systems that need a **quick price estimate + tier + sanity check** from text alone.


## 1. Setup



In [1]:
%pip install -q numpy gradio python-dotenv openai scikit-learn ipykernel

/Users/admin/Documents/ai_engineering/llm_engineering/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
from pathlib import Path

import numpy as np
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import mean_absolute_error

load_dotenv(override=True)

True

In [3]:
MODEL = "gpt-4o-mini"
EVAL_SIZE = 10  # items for batch evaluation

if os.environ.get("OPENROUTER_API_KEY"):
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
    print("Using OpenRouter")
else:
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
    print("Using OpenAI")

Using OpenRouter


## 2. Catalog loading

Load (description, price) pairs from `week6/human_out.csv` (last column = price). Used for band thresholds and evaluation.

In [4]:
def _split_text_price(row: str):
    last_comma = row.rfind(",")
    if last_comma == -1:
        return None
    text = row[:last_comma].strip().strip('"')
    try:
        p = float(row[last_comma + 1 :].strip())
    except ValueError:
        return None
    return (text, p)


def load_catalog():
    for path in [
        Path("week6/human_out.csv"),
        Path("human_out.csv"),
        Path("../human_out.csv"),
        Path("../../human_out.csv"),
    ]:
        if path.exists():
            out = []
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parsed = _split_text_price(line)
                    if parsed:
                        out.append({"text": parsed[0], "price": parsed[1]})
            if out:
                print(f"Catalog: {len(out)} items from {path}")
                return out
    # Fallback
    out = [
        {"text": "Guitar pedal, distortion, 9V", "price": 89.0},
        {"text": "Wireless mouse, ergonomic, USB", "price": 24.0},
        {"text": "Mechanical keyboard, RGB, Cherry MX", "price": 120.0},
    ]
    print(f"Using fallback catalog: {len(out)} items")
    return out


catalog = load_catalog()
prices_arr = np.array([x["price"] for x in catalog])

Catalog: 100 items from ../../human_out.csv


## 3. Price bands

From catalog prices we define **Budget** (≤33rd pct), **Mid** (33rd–66th), **Premium** (>66th). Used to label estimates and compute band accuracy.

In [5]:
Q33 = float(np.percentile(prices_arr, 33))
Q66 = float(np.percentile(prices_arr, 66))


def price_to_band(price: float) -> str:
    if price <= Q33:
        return "Budget"
    if price <= Q66:
        return "Mid-range"
    return "Premium"


print(f"Bands: Budget ≤ {Q33:.0f}, Mid ≤ {Q66:.0f}, else Premium")

Bands: Budget ≤ 34, Mid ≤ 77, else Premium


## 4. Valuation (LLM)

One call per item: ask for a **single number** (typical selling price in USD). Parse the reply and assign a band + short verdict.

In [6]:
def _first_number(s: str) -> float:
    if s is None:
        return 0.0
    s = str(s).replace(",", "").replace(" ", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0


def estimate_value(description: str):
    """Returns (price_estimate, band, verdict)."""
    prompt = (
        "You are a product valuation assistant. Based only on the following product description, "
        "give your best guess of its typical selling price in US dollars. "
        "Reply with exactly one number, no symbols or explanation.\n\n"
        f"Description:\n{description}\n\nEstimated price (USD):"
    )
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        raw = r.choices[0].message.content
        price = _first_number(raw)
    except Exception as e:
        print(f"API error: {e}")
        price = 0.0
    band = price_to_band(price)
    if price <= 0:
        verdict = "Unable to estimate"
    elif band == "Budget":
        verdict = "Likely budget-friendly"
    elif band == "Premium":
        verdict = "Likely premium; check similar listings"
    else:
        verdict = "Likely fair for mid-range"
    return (price, band, verdict)

## 5. Baselines

- **Median**: always predict catalog median.
- **Random**: random draw from catalog price range (for variability check).

In [7]:
median_price = float(np.median(prices_arr))
p_min, p_max = float(prices_arr.min()), float(prices_arr.max())


def baseline_median(_text):
    return median_price


def baseline_random(_text):
    return float(np.random.uniform(p_min, p_max))


print(f"Median baseline: {median_price:.0f} USD; range [{p_min:.0f}, {p_max:.0f}]")

Median baseline: 50 USD; range [5, 550]


## 6. Evaluation

On the first **N** catalog items we compute: **MAE**, **MAPE** (mean absolute % error), and **band accuracy** (fraction of correct Budget/Mid/Premium labels).

In [8]:
def run_eval(data, predictor, n=EVAL_SIZE):
    subset = data[:n]
    y_true = [x["price"] for x in subset]
    y_pred = [predictor(x["text"]) for x in subset]
    mae = mean_absolute_error(y_true, y_pred)
    # MAPE: avoid div by zero
    errs = [abs(t - p) / t if t > 0 else 0 for t, p in zip(y_true, y_pred)]
    mape = 100.0 * np.mean(errs) if errs else 0.0
    bands_true = [price_to_band(t) for t in y_true]
    bands_pred = [price_to_band(p) for p in y_pred]
    band_acc = sum(a == b for a, b in zip(bands_true, bands_pred)) / len(bands_true) if bands_true else 0.0
    return mae, mape, band_acc, y_true, y_pred


n = EVAL_SIZE
mae_m, mape_m, acc_m, _, _ = run_eval(catalog, baseline_median, n)
print(f"Median baseline (n={n}) — MAE: {mae_m:.1f}  MAPE: {mape_m:.1f}%  Band acc: {acc_m:.2%}")

Median baseline (n=10) — MAE: 62.5  MAPE: 78.2%  Band acc: 30.00%


In [9]:
mae_l, mape_l, acc_l, truths, guesses = run_eval(catalog, lambda t: estimate_value(t)[0], n)
print(f"LLM valuation (n={n}) — MAE: {mae_l:.1f}  MAPE: {mape_l:.1f}%  Band acc: {acc_l:.2%}")
print("Sample (true vs pred):")
for i in range(min(5, n)):
    print(f"  {truths[i]:.0f} vs {guesses[i]:.0f}")

LLM valuation (n=10) — MAE: 73.0  MAPE: 163.2%  Band acc: 50.00%
Sample (true vs pred):
  120 vs 150
  300 vs 150
  43 vs 50
  55 vs 25
  12 vs 50


## 7. Gradio app

Single page: enter a product description (or pick one from the catalog). Output: **estimated price (USD)**, **band**, and **verdict**. Optional: "Evaluate on sample" runs the LLM on the first N items and shows MAPE and band accuracy.

In [10]:
def ui_estimate(desc: str):
    if not (desc or desc.strip()):
        return "Enter or select a product description.", "", ""
    price, band, verdict = estimate_value(desc.strip())
    return f"{price:,.0f} USD", band, verdict


def ui_batch_eval():
    mae, mape, acc, _, _ = run_eval(catalog, lambda t: estimate_value(t)[0], EVAL_SIZE)
    return f"MAE: {mae:.1f}  |  MAPE: {mape:.1f}%  |  Band accuracy: {acc:.2%}"


catalog_choices = [x["text"] for x in catalog[:50]]

def do_estimate(text: str, selected: str):
    desc = (text or "").strip() or (selected or "").strip()
    return ui_estimate(desc)

with gr.Blocks(title="Product Valuation & Deal Checker") as app:
    gr.Markdown("## Product Valuation & Deal Checker")
    with gr.Row():
        inp = gr.Textbox(
            label="Product description",
            placeholder="Paste or type product title + details…",
            lines=4,
        )
    gr.Markdown("Or pick a sample from the catalog:")
    dropdown = gr.Dropdown(choices=catalog_choices, label="Catalog sample", allow_custom_value=False)
    btn_est = gr.Button("Estimate")
    out_price = gr.Textbox(label="Estimated price (USD)")
    out_band = gr.Textbox(label="Price band")
    out_verdict = gr.Textbox(label="Verdict")
    btn_est.click(fn=do_estimate, inputs=[inp, dropdown], outputs=[out_price, out_band, out_verdict])
    gr.Markdown("---")
    btn_batch = gr.Button("Run batch evaluation (first N items)")
    out_batch = gr.Textbox(label="Batch results")
    btn_batch.click(fn=ui_batch_eval, inputs=[], outputs=[out_batch])

app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---

**How to run:** Set `OPENAI_API_KEY` or `OPENROUTER_API_KEY` in `.env`, then run all cells. The last cell starts the Gradio app. Use "Estimate" for a single product; use "Run batch evaluation" to see MAPE and band accuracy on the catalog sample.